# PYNQ 通用模型硬件推理测试

自动从 `model_info.json` 读取网络结构，支持 MLP / LeNet-5 / 任意 Conv+FC+Pool 组合。

**前提：** 宿主机已运行 `04_generate_model_data.ipynb`，生成 `<arch>_data.zip`

## 0. 选择数据目录

In [ ]:
# ===== 修改这一行即可切换模型 =====
DATA_DIR = 'lenet5_data'    # 'mlp_data' or 'lenet5_data'
# ==================================

In [ ]:
!unzip -o {DATA_DIR}.zip

In [ ]:
from pynq import Overlay, MMIO
import numpy as np
import os, glob, json, time

## 1. 加载 Bitstream + CSR

In [ ]:
ol = Overlay('cim_soc.bit')
print('Overlay loaded!')
BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)

In [ ]:
CTRL=0x000; STATUS=0x004; CSR_IN_DIM=0x010; CSR_OUT_DIM=0x014
CSR_N_IB=0x018; CSR_N_OB=0x01C; REQUANT_MULT=0x020; REQUANT_SHIFT=0x024
INPUT_ZP=0x028; ACT_MODE=0x02C; CYCLE_CNT_LO=0x030; MAC_CNT_LO=0x038
PRED_CLASS=0x040; WDMA_ADDR=0x044; WDMA_DATA=0x048; WDMA_CTRL=0x04C
LOGIT_BASE=0x100; MEM_INPUT=0x1000; MEM_BIAS=0x2000
TILE_ROWS=16; TILE_COLS=16; ELEMS_PER_CHUNK=4; CHUNKS_PER_ROW=4; CHUNKS_PER_TILE=64

## 2. 硬件操作 + im2col + pool

In [ ]:
def soft_reset(): mmio.write(CTRL, 0x4)
def clear_done(): mmio.write(CTRL, 0x2)

def configure(in_dim, out_dim, zp, mult, shift, relu):
    mmio.write(CSR_IN_DIM, in_dim); mmio.write(CSR_OUT_DIM, out_dim)
    mmio.write(CSR_N_IB, (in_dim+15)//16); mmio.write(CSR_N_OB, (out_dim+15)//16)
    mmio.write(REQUANT_MULT, mult); mmio.write(REQUANT_SHIFT, shift)
    mmio.write(INPUT_ZP, int(zp)&0xFFFFFFFF); mmio.write(ACT_MODE, 1 if relu else 0)

def load_weights(chunks):
    mmio.write(WDMA_ADDR, 0); mmio.write(WDMA_CTRL, 0x02)
    for c in chunks: mmio.write(WDMA_DATA, int(c))
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bl):
    for i,b in enumerate(bl): mmio.write(MEM_BIAS+4*i, int(b)&0xFFFFFFFF)

def load_input(d):
    p = list(d)
    while len(p)%16: p.append(0)
    for i,x in enumerate(p): mmio.write(MEM_INPUT+4*i, int(x)&0xFF)

def run_hw():
    clear_done(); mmio.write(CTRL, 0x1)
    while not (mmio.read(STATUS)&0x2): pass
    return mmio.read(CYCLE_CNT_LO)

def read_out(n):
    return np.array([np.uint8(mmio.read(LOGIT_BASE+4*i)&0xFF).view(np.int8) for i in range(n)], dtype=np.int8)

def hw_mvm(x_u8, wc, bu, out_dim, zp, mult, shift, relu):
    soft_reset(); configure(len(x_u8), out_dim, zp, mult, shift, relu)
    load_weights(wc); load_bias(bu); load_input(x_u8)
    cyc = run_hw()
    return read_out(out_dim), cyc

def im2col(feat, kh, kw, stride=1, padding=0):
    C,H,W = feat.shape
    if padding>0:
        p=np.zeros((C,H+2*padding,W+2*padding),dtype=feat.dtype)
        p[:,padding:padding+H,padding:padding+W]=feat; feat=p
    Hp,Wp=feat.shape[1],feat.shape[2]
    oh=(Hp-kh)//stride+1; ow=(Wp-kw)//stride+1
    col=np.zeros((C*kh*kw,oh*ow),dtype=feat.dtype); idx=0
    for i in range(oh):
        for j in range(ow):
            col[:,idx]=feat[:,i*stride:i*stride+kh,j*stride:j*stride+kw].flatten(); idx+=1
    return col,oh,ow

def maxpool(feat,k=2,s=2):
    C,H,W=feat.shape; oh,ow=H//s,W//s
    out=np.zeros((C,oh,ow),dtype=feat.dtype)
    for c in range(C):
        for i in range(oh):
            for j in range(ow): out[c,i,j]=feat[c,i*s:i*s+k,j*s:j*s+k].max()
    return out

print('Functions loaded.')

## 3. 加载模型参数

In [ ]:
# Read model structure
with open(f'{DATA_DIR}/model_info.json') as f:
    model_info = json.load(f)
arch = model_info['arch']
layer_defs = model_info['layers']
print(f'Architecture: {arch}')
for l in layer_defs:
    print(f"  {l['name']}: {l['type']}")

# Read hex files
def rhex(p):
    with open(p) as f: return [int(l.strip(),16) for l in f if l.strip()]

qp_vals = rhex(f'{DATA_DIR}/quant_params.hex')
zp_vals = rhex(f'{DATA_DIR}/zero_points.hex')

compute_layers = [l for l in layer_defs if l['type'] in ('conv','fc')]
hw_layers = {}
for i, l in enumerate(compute_layers):
    name = l['name']
    hw_layers[name] = {
        'wc': rhex(f'{DATA_DIR}/{name}_weight_tiles.hex'),
        'bu': rhex(f'{DATA_DIR}/{name}_bias.hex'),
        'mult': qp_vals[i*2], 'shift': qp_vals[i*2+1],
        'zp': int(np.int32(np.uint32(zp_vals[i]))),
    }
    print(f"  {name}: {len(hw_layers[name]['wc'])} chunks, mult={hw_layers[name]['mult']}")

## 4. AXI 测试

In [ ]:
soft_reset()
mmio.write(CSR_IN_DIM, 784)
rb = mmio.read(CSR_IN_DIM)
print(f'AXI test: {"PASS" if rb==784 else "FAIL"}')
assert rb == 784

## 5. 通用推理函数

In [ ]:
def hw_infer(img_u8_flat, layer_defs, hw_layers):
    """Run full network on hardware. Returns (pred, final_output, total_cycles)."""
    arch = model_info['arch']
    x = img_u8_flat.reshape(1,28,28) if arch != 'mlp' else img_u8_flat
    total_cyc = 0

    for ldef in layer_defs:
        name = ldef['name']
        if ldef['type'] == 'conv':
            hl = hw_layers[name]
            C_out = ldef['C_out']
            col, oh, ow = im2col(x, ldef['K_h'], ldef['K_w'], ldef['stride'], ldef['padding'])
            out = np.zeros((C_out, oh*ow), dtype=np.int8)
            for p in range(oh*ow):
                out[:,p], cyc = hw_mvm(col[:,p].tolist(), hl['wc'], hl['bu'],
                                        C_out, hl['zp'], hl['mult'], hl['shift'], ldef['relu'])
                total_cyc += cyc
            x = out.reshape(C_out, oh, ow)

        elif ldef['type'] == 'pool':
            x = maxpool(x.view(np.int8), ldef['kernel'], ldef['stride']).view(np.uint8)

        elif ldef['type'] == 'fc':
            hl = hw_layers[name]
            if x.ndim > 1: x = x.flatten()
            if hasattr(x,'dtype') and x.dtype == np.int8: x = x.view(np.uint8)
            x, cyc = hw_mvm(x.tolist() if isinstance(x,np.ndarray) else list(x),
                            hl['wc'], hl['bu'], ldef['out_dim'],
                            hl['zp'], hl['mult'], hl['shift'], ldef['relu'])
            total_cyc += cyc

    final = np.array(x).flatten() if isinstance(x, np.ndarray) else np.array(x)
    return int(np.argmax(final)), final, total_cyc

print('Inference function ready.')

## 6. 逐张测试

In [ ]:
img_dir = f'{DATA_DIR}/test_images'
img_files = sorted(glob.glob(f'{img_dir}/img_????.hex'))
n = len(img_files)
print(f'Found {n} test images\n')

# Find golden output layer name
last_compute = [l['name'] for l in layer_defs if l['type'] in ('conv','fc')][-1]

hw_correct = 0; bit_exact = 0; total = 0; errors = []
t0 = time.time()

for path in img_files:
    name = os.path.basename(path).replace('.hex','')
    img_u8 = np.array(rhex(path), dtype=np.uint8)
    label = int(open(f'{img_dir}/{name}_label.txt').read().strip())
    py_pred = int(open(f'{img_dir}/{name}_pred.txt').read().strip())
    golden = np.array([np.uint8(v).view(np.int8) for v in rhex(f'{img_dir}/{name}_{last_compute}.hex')], dtype=np.int8)

    hw_pred, hw_final, cyc = hw_infer(img_u8, layer_defs, hw_layers)
    hw_final_i8 = hw_final.view(np.int8) if hw_final.dtype != np.int8 else hw_final

    ok = np.array_equal(hw_final_i8, golden) and (hw_pred == py_pred)
    if hw_pred == label: hw_correct += 1
    if ok: bit_exact += 1
    else: errors.append(name)
    total += 1

    m = '\u2713' if ok else '\u2717'
    c = '\u2713' if hw_pred==label else '\u2717'
    print(f'  {name}: label={label} hw={hw_pred} py={py_pred} exact={m} correct={c} ({cyc} cyc)')

    if not ok and not np.array_equal(hw_final_i8, golden):
        diffs = np.where(hw_final_i8 != golden)[0]
        print(f'    Output diffs at: {diffs[:5].tolist()}')

elapsed = time.time() - t0
print(f'\nWall time: {elapsed:.1f}s ({elapsed/total:.2f}s/img)')

## 7. 结果总结

In [ ]:
print('='*60)
print(f'{arch.upper()} Hardware Inference Results')
print('='*60)
print(f'  Architecture:      {arch}')
print(f'  Total images:      {total}')
print(f'  HW accuracy:       {100.*hw_correct/total:.1f}% ({hw_correct}/{total})')
print(f'  Bit-exact match:   {100.*bit_exact/total:.1f}% ({bit_exact}/{total})')
print(f'  Errors:            {len(errors)}')
print()
if len(errors) == 0:
    print(f'>>> ALL {total} IMAGES BIT-EXACT MATCH <<<')
    print(f'>>> CIM SoC correctly runs {arch.upper()} on real MNIST! <<<')
else:
    print(f'  Mismatched: {errors}')
print('='*60)